# Driving School Installs Prediction — ARIMA Family

Unified notebook combining **ARIMA**, **SARIMA**, and **SARIMAX** models.
Shared preprocessing runs once; each model's parameter search, fitting,
rolling evaluation, and saving are kept in separate sections.

## Imports

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error
from utils import check_stationarity

warnings.filterwarnings("ignore")

## Hyperparameters & Config

In [ ]:
DATA_CSV_FILE   = "../../../data/ds_data_clean.csv"
TARGET_COL      = "Daily User Installs"
EXOG_COLS       = [
    "Daily Device Uninstalls",
    # "User to Device Ratio",
    "Daily User Uninstalls",
    "Active Device Installs",
    "Daily Average Rating",
    "Daily Crashes",
    "Daily ANRs",
    "Is Weekend",
]
TRAIN_RATIO     = 0.8
SEASONAL_PERIOD = 7   # Weekly seasonality
ROLL_PERIOD     = 7   # Rolling forecast window (days)

## Loading Data

In [ ]:
df = pd.read_csv(DATA_CSV_FILE)
df["Date"] = pd.to_datetime(df["Date"])
df["Is Weekend"] = (df["Date"].dt.dayofweek >= 5).astype(int)  # 5=Sat, 6=Sun
df.set_index("Date", inplace=True)
df.sort_index(inplace=True)
df.head()

## Stationarity Check

In [ ]:
check_stationarity(df[TARGET_COL], title=TARGET_COL)

## Preprocessing and Temporal Split

In [ ]:
train_size = int(len(df) * TRAIN_RATIO)
train_df = df.iloc[:train_size]
test_df  = df.iloc[train_size:]

# Univariate series (ARIMA / SARIMA)
train = train_df[TARGET_COL]
test  = test_df[TARGET_COL]

# Multivariate split (SARIMAX)
y_train = train_df[TARGET_COL]
y_test  = test_df[TARGET_COL]
X_train = train_df[EXOG_COLS]
X_test  = test_df[EXOG_COLS]

print(f"Train: {train_df.index[0].date()} → {train_df.index[-1].date()}  ({len(train)} days)")
print(f"Test:  {test_df.index[0].date()} → {test_df.index[-1].date()}  ({len(test)} days)")

## Rolling Forecast Helper

Walk-forward evaluation shared by all three models. At each chunk the model
is refit on all history seen so far, forecasts `ROLL_PERIOD` steps ahead,
then the true values are revealed before the next chunk.

In [ ]:
def rolling_forecast(fit_fn, endog_train, endog_test,
                     exog_train=None, exog_test=None,
                     roll_period=ROLL_PERIOD):
    """
    Walk-forward (rolling) validation.

    Parameters
    ----------
    fit_fn      : callable(history_y, history_X, steps, chunk_X) -> array
                  Fits the model on current history and returns forecasts.
    endog_train : pd.Series  — training target
    endog_test  : pd.Series  — test target
    exog_train  : pd.DataFrame | None — training exog (SARIMAX only)
    exog_test   : pd.DataFrame | None — test exog (SARIMAX only)
    roll_period : int — chunk size (days per re-fit)

    Returns
    -------
    pd.Series of rolling predictions aligned to endog_test.index
    """
    history_y = list(endog_train.copy())
    history_X = exog_train.copy() if exog_train is not None else None
    preds = []

    for start in range(0, len(endog_test), roll_period):
        chunk_y = endog_test.iloc[start : start + roll_period]
        chunk_X = exog_test.iloc[start : start + roll_period] if exog_test is not None else None

        yhats = fit_fn(history_y, history_X, len(chunk_y), chunk_X)
        preds.extend(yhats)

        history_y.extend(chunk_y.tolist())
        if history_X is not None:
            history_X = pd.concat([history_X, chunk_X])

    return pd.Series(preds, index=endog_test.index)


def evaluate_and_plot(name, actual, preds, color, roll_period=ROLL_PERIOD):
    """Print RMSE/MAE and plot actual vs rolling forecast."""
    rmse = np.sqrt(mean_squared_error(actual, preds))
    mae  = mean_absolute_error(actual, preds)

    print("-" * 35)
    print(f"{name} ROLLING FORECAST RESULTS")
    print(f"  (re-fit every {roll_period} days)")
    print("-" * 35)
    print(f"RMSE: {rmse:.2f} installs")
    print(f"MAE:  {mae:.2f} installs")
    print("-" * 35)

    plt.figure(figsize=(14, 5))
    plt.plot(actual.index, actual, label="Actual", color="royalblue", alpha=0.8)
    plt.plot(actual.index, preds,  label=f"Rolling Forecast ({roll_period}d)",
             color=color, linestyle="--")
    plt.title(f"Daily Install Prediction — {name} (Rolling {roll_period}-Day)")
    plt.xlabel("Date")
    plt.ylabel("Daily User Installs")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return rmse, mae


def plot_residuals(name, actual, preds):
    """Histogram of forecast residuals."""
    residuals = actual - preds
    plt.figure(figsize=(10, 5))
    plt.hist(residuals, bins=50, color="lightcoral", edgecolor="black")
    plt.axvline(0, color="black", linestyle="--")
    plt.title(f"{name} Rolling Forecast Residuals (Actual - Predicted)")
    plt.show()


def save_model(model_fit, model_path, meta_path, meta):
    """Save fitted statsmodels result and metadata JSON."""
    model_fit.save(model_path)
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"Model saved    → {model_path}")
    print(f"Metadata saved → {meta_path}")

---
# Model 1: ARIMA
Univariate, non-seasonal baseline.

## ARIMA — Parameter Search

In [ ]:
arima_auto = auto_arima(
    train,
    seasonal=False,
    trace=True,
    error_action="ignore",
    suppress_warnings=True,
    stepwise=True,
)
print(arima_auto.summary())

## ARIMA — Fit

In [ ]:
ap, ad, aq = arima_auto.order
arima_model = ARIMA(train, order=(ap, ad, aq))
arima_fit   = arima_model.fit()
print(arima_fit.summary())

## ARIMA — Rolling Forecast & Evaluation

In [ ]:
def _arima_fit_fn(history_y, history_X, steps, chunk_X):
    m = ARIMA(history_y, order=(ap, ad, aq)).fit()
    return m.forecast(steps=steps)

arima_preds          = rolling_forecast(_arima_fit_fn, train, test)
arima_rmse, arima_mae = evaluate_and_plot("ARIMA", test, arima_preds, color="red")

## ARIMA — Residual Analysis

In [ ]:
plot_residuals("ARIMA", test, arima_preds)

## ARIMA — Save Model

In [ ]:
save_model(
    arima_fit,
    "arima_model.pkl",
    "arima_model_meta.json",
    meta={
        "model":            "ARIMA",
        "order":            list(arima_auto.order),
        "target_col":       TARGET_COL,
        "train_ratio":      TRAIN_RATIO,
        "forecast_type":    "rolling_nstep",
        "roll_period_days": ROLL_PERIOD,
        "train_start":      str(train_df.index[0].date()),
        "train_end":        str(train_df.index[-1].date()),
        "test_start":       str(test_df.index[0].date()),
        "test_end":         str(test_df.index[-1].date()),
        "rmse":             round(arima_rmse, 4),
        "mae":              round(arima_mae,  4),
    }
)

---
# Model 2: SARIMA
Univariate with weekly seasonality.

## SARIMA — Parameter Search

In [ ]:
sarima_auto = auto_arima(
    train,
    seasonal=True,
    m=SEASONAL_PERIOD,
    trace=True,
    error_action="ignore",
    suppress_warnings=True,
    stepwise=True,
)
print(sarima_auto.summary())

## SARIMA — Fit

In [ ]:
sp, sd, sq = sarima_auto.order
sP, sD, sQ, ss = sarima_auto.seasonal_order

print(f"Fitting SARIMA({sp},{sd},{sq})x({sP},{sD},{sQ},{ss})")

sarima_model = SARIMAX(
    train,
    order=(sp, sd, sq),
    seasonal_order=(sP, sD, sQ, ss),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False)
print(sarima_fit.summary())

## SARIMA — Rolling Forecast & Evaluation

In [ ]:
def _sarima_fit_fn(history_y, history_X, steps, chunk_X):
    m = SARIMAX(
        history_y,
        order=(sp, sd, sq),
        seasonal_order=(sP, sD, sQ, ss),
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)
    return m.forecast(steps=steps)

sarima_preds            = rolling_forecast(_sarima_fit_fn, train, test)
sarima_rmse, sarima_mae = evaluate_and_plot("SARIMA", test, sarima_preds, color="crimson")

## SARIMA — Residual Analysis

In [ ]:
plot_residuals("SARIMA", test, sarima_preds)

## SARIMA — Save Model

In [ ]:
save_model(
    sarima_fit,
    "sarima_model.pkl",
    "sarima_model_meta.json",
    meta={
        "model":            "SARIMA",
        "order":            list(sarima_auto.order),
        "seasonal_order":   list(sarima_auto.seasonal_order),
        "seasonal_period":  SEASONAL_PERIOD,
        "target_col":       TARGET_COL,
        "train_ratio":      TRAIN_RATIO,
        "forecast_type":    "rolling_nstep",
        "roll_period_days": ROLL_PERIOD,
        "train_start":      str(train_df.index[0].date()),
        "train_end":        str(train_df.index[-1].date()),
        "test_start":       str(test_df.index[0].date()),
        "test_end":         str(test_df.index[-1].date()),
        "rmse":             round(sarima_rmse, 4),
        "mae":              round(sarima_mae,  4),
    }
)

---
# Model 3: SARIMAX
Multivariate — seasonal ARIMA with exogenous features.

## SARIMAX — Parameter Search

In [ ]:
sarimax_auto = auto_arima(
    y_train,
    X=X_train,
    seasonal=True,
    m=SEASONAL_PERIOD,
    trace=True,
    error_action="ignore",
    suppress_warnings=True,
    stepwise=True,
)
print(sarimax_auto.summary())

## SARIMAX — Fit

In [ ]:
xp, xd, xq = sarimax_auto.order
xP, xD, xQ, xs = sarimax_auto.seasonal_order

sarimax_model = SARIMAX(
    y_train,
    exog=X_train,
    order=(xp, xd, xq),
    seasonal_order=(xP, xD, xQ, xs),
)
sarimax_fit = sarimax_model.fit(disp=False)
print(sarimax_fit.summary())

## SARIMAX — Rolling Forecast & Evaluation

In [ ]:
def _sarimax_fit_fn(history_y, history_X, steps, chunk_X):
    m = SARIMAX(
        history_y,
        exog=history_X,
        order=(xp, xd, xq),
        seasonal_order=(xP, xD, xQ, xs),
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)
    return m.forecast(steps=steps, exog=chunk_X)

sarimax_preds              = rolling_forecast(_sarimax_fit_fn, y_train, y_test,
                                              exog_train=X_train, exog_test=X_test)
sarimax_rmse, sarimax_mae = evaluate_and_plot("SARIMAX", y_test, sarimax_preds, color="green")

## SARIMAX — Residual Analysis

In [ ]:
plot_residuals("SARIMAX", y_test, sarimax_preds)

## SARIMAX — Save Model

In [ ]:
save_model(
    sarimax_fit,
    "sarimax_model.pkl",
    "sarimax_model_meta.json",
    meta={
        "model":            "SARIMAX",
        "order":            list(sarimax_auto.order),
        "seasonal_order":   list(sarimax_auto.seasonal_order),
        "seasonal_period":  SEASONAL_PERIOD,
        "target_col":       TARGET_COL,
        "exog_cols":        EXOG_COLS,
        "train_ratio":      TRAIN_RATIO,
        "forecast_type":    "rolling_nstep",
        "roll_period_days": ROLL_PERIOD,
        "train_start":      str(train_df.index[0].date()),
        "train_end":        str(train_df.index[-1].date()),
        "test_start":       str(test_df.index[0].date()),
        "test_end":         str(test_df.index[-1].date()),
        "rmse":             round(sarimax_rmse, 4),
        "mae":              round(sarimax_mae,  4),
    }
)

---
# Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["ARIMA", "SARIMA", "SARIMAX"],
    "RMSE":  [arima_rmse,   sarima_rmse,   sarimax_rmse],
    "MAE":   [arima_mae,    sarima_mae,    sarimax_mae],
}).set_index("Model")

print("=" * 35)
print("      MODEL COMPARISON SUMMARY")
print("=" * 35)
print(results.to_string(float_format="{:.2f}".format))
print("=" * 35)
print(f"Best RMSE → {results["RMSE"].idxmin()}")
print(f"Best MAE  → {results["MAE"].idxmin()}")

# Side-by-side bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
results["RMSE"].plot(kind="bar", ax=axes[0], color=["red","crimson","green"], edgecolor="black")
axes[0].set_title("RMSE by Model")
axes[0].set_ylabel("RMSE (installs)")
axes[0].tick_params(axis="x", rotation=0)

results["MAE"].plot(kind="bar", ax=axes[1], color=["red","crimson","green"], edgecolor="black")
axes[1].set_title("MAE by Model")
axes[1].set_ylabel("MAE (installs)")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()